# 第8回 異常データの検出

**データ分析（da）／ 情報工学部 情報工学科 2年次 前期 ／ 担当: 後藤 弘光**

---

第4回ではデータの汚れとして「欠損・重複・異常値」の3種類を学び、異常値については `quantity > 100` のような **人が決めた閾値** で抽出しました。今回はさらに一歩進んで、**データ自身の分布から客観的な基準を作って異常値を検出する**方法を学びます。

## 1. 要点まとめ

### 1.1 異常値（外れ値）とは

**異常値 (outlier)** は、データの大多数から大きく離れた極端な値です。異常値には 2 つの顔があります。

- **無視すべきノイズ**: 入力ミス、単位の間違い、測定機器の故障 → 除去して分析する
- **発見すべきチャンス**: 不正取引、故障の前兆、優良顧客 → むしろ抽出して深掘りする

どちらにしても、まず「どれが外れ値か」を**客観的な基準**で特定する必要があります。

### 1.2 四分位数と IQR

データを小さい順に並べたとき、

- **第1四分位数 (Q1)**: 下から 25% の位置の値
- **第3四分位数 (Q3)**: 下から 75% の位置の値
- **四分位範囲 (IQR)** = Q3 − Q1 （データの中央 50% が収まる幅）

### 1.3 IQR 法による外れ値の定義

広く使われている基準が **IQR 法**（テューキーの基準）です。

$$
\text{下限} = Q_1 - 1.5 \times \text{IQR}, \qquad \text{上限} = Q_3 + 1.5 \times \text{IQR}
$$

この範囲から外れた値を外れ値とみなします。データの単位や分布に依存しない汎用的な基準で、箱ひげ図の「ひげ」の長さもこの定義に基づいています。

### 1.4 Pandas で使う関数

| 関数 | 役割 |
| :-- | :-- |
| `df['列'].quantile(q)` | 指定した分位点の値を返す（`q=0.25` で Q1） |
| `df[df['列'] > 上限]` | ブール索引で上限を超える行を抽出 |
| `df[(df['列'] > 上限) \| (df['列'] < 下限)]` | 上下両方の外れ値を抽出 |

### 1.5 分布と IQR を目で見る

下のヒストグラムと箱ひげ図は、ある仮想データについて IQR 法がどのように外れ値を切り出すかを示したものです。ヒストグラムの縦線が Q1・中央値・Q3、赤い破線が IQR 法による上下限です。箱ひげ図の「ひげ」の長さもこの上下限に基づいており、両者は同じ基準を別の角度から見せています。

> 箱ひげ図そのものは第10回「データの可視化」で改めて学びますが、ここでは IQR が視覚的にどこを指しているかだけを感じ取ってください。

In [ ]:
# 分布と IQR を可視化する（デモ用の仮想データ）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
sample = np.concatenate([rng.normal(50, 10, 200), [95, 100, 110]])  # 末尾に外れ値

q1, q3 = np.quantile(sample, [0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# 左: ヒストグラム + Q1/中央値/Q3/上下限
axes[0].hist(sample, bins=30, color='#9ecae1', edgecolor='white')
for x, label, color in [
    (q1, 'Q1', 'black'),
    (np.median(sample), 'median', 'gray'),
    (q3, 'Q3', 'black'),
    (upper, 'upper', 'red'),
    (lower, 'lower', 'red'),
]:
    axes[0].axvline(x, color=color, linestyle='--' if color == 'red' else '-')
    axes[0].text(x, axes[0].get_ylim()[1]*0.95, label, rotation=90, va='top', ha='right', fontsize=9, color=color)
axes[0].set_title('histogram with IQR')
axes[0].set_xlabel('value')

# 右: 箱ひげ図
axes[1].boxplot(sample, vert=False)
axes[1].set_title('box plot')
axes[1].set_xlabel('value')

plt.tight_layout()
plt.show()

---

## 2. 演習

### 2.1 背景

> 先月のオンラインショップの購入金額データを見ていたら、数人だけ他の顧客とは明らかに桁違いの金額を使っている人がいるようです。キャンペーンの効果測定で平均購入額を出したいのですが、こうした極端な顧客をどう扱うか決める前に、まず「誰が、どれくらい外れているのか」を客観的に把握したいです。

配布データ `purchases.csv` は、ある月のオンラインショップの顧客別購入金額（500 件、`customer_id, amount`）です。

#### 問い

IQR 法（上限 = Q3 + 1.5×IQR、下限 = Q1 − 1.5×IQR）に基づいて、`amount` 列の外れ値に該当する顧客を特定し、その件数と最大額を把握するには、どのような手順が必要か？

#### 仮説

`quantile(0.25)` と `quantile(0.75)` で Q1・Q3 を求め、IQR を計算して上下限を決め、ブール索引で上限を超える行や下限を下回る行を抽出すれば、客観的な基準で外れ値を特定できるはずです。

### 2.2 分析

In [ ]:
# ============================================================
# 設計 → 実装
# ------------------------------------------------------------
# データ分析は「いきなりコードを書く」ものではありません。
# まず日本語で分析の骨組み（手順）を書き出し、
# それを一つずつ Python コードに翻訳していきます。
#
# このセルには授業中に以下の流れで記入します:
#   1. 骨組みコメントを書く（# 1. …, # 2. …, # 3. …）
#      ← 「何をするか」を日本語で明文化
#   2. 各コメントの下にコードを追記する
#      ← 日本語の手順を Python に「翻訳」
#
# 生成 AI も、骨組みが明確なほど正確なコードを返してくれます。
# この「描いてから書く」習慣を本科目で身につけましょう。
# ============================================================

# 1.

# 2.

# 3.

# 4.

# 5.

# 6.


### 2.3 サマリー

以下の穴埋め項目を、演習で実行したコードの出力結果をもとに記入してください。

#### 分析結果の確認

- Q1（第1四分位数）は約 \_\_\_\_ 円でした
- Q3（第3四分位数）は約 \_\_\_\_ 円でした
- IQR は約 \_\_\_\_ 円でした
- IQR 法による上限は約 \_\_\_\_ 円、下限は約 \_\_\_\_ 円でした
- 外れ値に該当する顧客の件数は \_\_\_\_ 件でした
- 外れ値のうち最大の購入金額は \_\_\_\_ 円でした

#### 結論

上記の確認により、`quantile()` で \_\_\_\_ を求め、ブール索引による条件抽出を組み合わせることで、データ自身の分布から客観的な基準で \_\_\_\_ を特定できることを確認できました。これらの値が「除去すべきノイズ」なのか「発見すべきチャンス」なのかは、ドメイン知識と合わせて判断する必要があります。